# Random Guide-Selection Strategies

Compares different random strategies for selecting guide sets.
Each strategy gets its own run directory matched by name pattern.

### TODOs
- Add best single guide comparison (k=1 vs k>1)
- Revisit metrics — consider adding new ones
- Egraph growth curves from `top_k.json` per-iteration data (nodes/classes over iterations, averaged across trials, by k) -> File is very big

In [ ]:
import json
import math

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from helpers import find_latest_run, load_top_k

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 8)})

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
# Add new strategies here: (display_name, run-directory pattern)
STRATEGIES = [
    ("count-full-union", "nodes-1000000000-timems-200-strategy-count-based-fullunion-true"),
    ("count-root-union", "nodes-1000000000-timems-200-strategy-count-based-fullunion-false"),
    ("naive-full-union", "nodes-1000000000-timems-200-strategy-naive-fullunion-true"),
    ("naive-root-union", "nodes-1000000000-timems-200-strategy-naive-fullunion-false"),
]

In [ ]:
# ── Load & parse ──────────────────────────────────────────────────────
frames_no, frames_wi = [], []
for name, pattern in STRATEGIES:
    run_dir = find_latest_run(pattern)
    print(f"{name}: {run_dir}")
    frame_no = load_top_k(run_dir, name, with_replacement=False)
    frame_wi = load_top_k(run_dir, name, with_replacement=True)
    print(f"  no-replacement: {len(frame_no)} rows, with-replacement: {len(frame_wi)} rows")
    frames_no.append(frame_no)
    frames_wi.append(frame_wi)


def _make_df(frames):
    return pl.concat(frames).with_columns(
        (pl.col("nodes") / pl.col("classes")).alias("nodes_per_class"),
    )


df_no = _make_df(frames_no)
df_with = _make_df(frames_wi)
DATASETS = [("no-replacement", df_no), ("with-replacement", df_with)]

k_values = sorted(df_no["k"].unique().to_list())
strat_names = [s[0] for s in STRATEGIES]

print(f"\nk values: {k_values}")
df_no.head(5)

In [ ]:
# ── Load baselines from stats.json ────────────────────────────────────
baselines = {}
for name, pattern in STRATEGIES:
    run_dir = find_latest_run(pattern)
    with open(run_dir / "stats.json") as f:
        all_stats = json.load(f)
    node_avg = sum(s["goal_egraph_nodes"] for s in all_stats) / len(all_stats)
    time_avg = sum(s["goal_eqsat_time"] for s in all_stats) / len(all_stats)
    iter_avg = sum(s["goal_egraph_iters"] for s in all_stats) / len(all_stats)
    classes_avg = sum(s["goal_egraph_classes"] for s in all_stats) / len(all_stats)
    baselines[name] = {
        "nodes": node_avg,
        "total_time": time_avg,
        "iters": iter_avg,
        "classes": classes_avg,
        "nodes_per_class": node_avg / classes_avg,
    }

print("Baselines (full eqsat, no guides):")
for name, vals in baselines.items():
    print(f"  {name}: nodes={vals['nodes']:.1f}, time={vals['total_time']:.6f}s")

## Metrics vs k

In [ ]:
PALETTE = plt.cm.tab10.colors  # type: ignore[attr-defined]
MARKER_CYCLE = ["o", "s", "D", "^", "v", "P", "X", "*"]

# For each metric: which aggregation is "best"?
METRIC_BEST_AGG = {
    "iters": "min",
    "nodes": "min",
    "classes": "min",
    "nodes_per_class": "max",
    "total_time": "min",
}


def strat_style(i):
    return {
        "color": PALETTE[i % len(PALETTE)],
        "marker": MARKER_CYCLE[i % len(MARKER_CYCLE)],
    }


# Metrics where a single shared baseline exists
BASELINE_METRICS = {"iters", "nodes", "classes", "nodes_per_class", "total_time"}

METRICS = ["iters", "nodes", "classes", "nodes_per_class", "total_time"]
n_metrics = len(METRICS)
n_cols = 3
n_rows = math.ceil(n_metrics / n_cols)

for _label, df in DATASETS:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
    axes_flat = axes.flat

    for metric, ax in zip(METRICS, axes_flat):
        best_agg = METRIC_BEST_AGG[metric]
        for i, strat in enumerate(strat_names):
            style = strat_style(i)
            strat_df = df.filter(pl.col("strategy") == strat)

            # Mean ± std across all trials
            agg = (
                strat_df.group_by("k")
                .agg(
                    pl.col(metric).mean().alias("mean"),
                    pl.col(metric).std().alias("std"),
                )
                .sort("k")
            )
            ks = agg["k"]
            means = agg["mean"]
            stds = agg["std"].fill_null(0)
            ax.plot(ks, means, label=strat, **style)
            ax.fill_between(ks, means - stds, means + stds, alpha=0.15, color=style["color"])

            # Best: for each seed×k take min/max over trials, then average across seeds
            per_seed_best = (
                strat_df.group_by("k", "seed")
                .agg(getattr(pl.col(metric), best_agg)().alias("seed_best"))
                .group_by("k")
                .agg(pl.col("seed_best").mean().alias("best"))
                .sort("k")
            )

            ax.plot(
                per_seed_best["k"],
                per_seed_best["best"],
                color=style["color"],
                linestyle="dotted",
                linewidth=1.5,
                label=f"{strat} best ({best_agg})",
            )

        # Single shared baseline (take from first strategy)
        if metric in BASELINE_METRICS:
            bval = baselines[strat_names[0]][metric]
            ax.axhline(
                bval, color="black", linestyle="--", linewidth=1.2, alpha=0.7, label="baseline"
            )

        ax.set_xlabel("k (number of guides)")
        metric_label = {
            "nodes": "nodes (incl. guide egraph)",
            "total_time": "total_time (incl. guide eqsat)",
        }.get(metric, metric)
        ax.set_ylabel(metric_label)
        ax.set_title(f"{metric_label} vs k (mean \u00b1 std)")
        ax.legend(fontsize=7)
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
        ax.set_xticks(k_values)

    # Hide any unused axes
    for ax in list(axes_flat)[n_metrics:]:
        ax.set_visible(False)

    fig.suptitle(_label, fontsize=14, y=1.01)
    fig.tight_layout()
    plt.show()

## Reachability rate vs k

In [ ]:
for _label, df in DATASETS:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for i, strat in enumerate(strat_names):
        style = strat_style(i)
        strat_df = df.filter(pl.col("strategy") == strat)

        rates = (
            strat_df.group_by("k")
            .agg(
                pl.col("iters").is_not_null().mean().alias("reached_rate"),
                pl.col("unreached").mean().alias("unreached_rate"),
                pl.col("not_enough_samples").mean().alias("insuf_rate"),
                # Conditional: reached / (reached + unreached), excluding sampling failures
                (
                    pl.col("iters").is_not_null().sum()
                    / (pl.col("not_enough_samples").not_()).sum()
                ).alias("cond_reach_rate"),
            )
            .sort("k")
        )
        ks = rates["k"]

        ax = axes[0]
        ax.plot(ks, rates["reached_rate"] * 100, label=f"{strat} reached", **style)
        ax.plot(
            ks,
            rates["cond_reach_rate"] * 100,
            color=style["color"],
            linestyle="dotted",
            linewidth=1.5,
            label=f"{strat} reached (excl. sampling failures)",
        )

        ax = axes[1]
        ax.plot(ks, rates["unreached_rate"] * 100, label=f"{strat} unreached", **style)
        ax.plot(
            ks,
            rates["insuf_rate"] * 100,
            color=style["color"],
            linestyle="dashed",
            linewidth=1.5,
            label=f"{strat} insufficient samples",
        )

    axes[0].set_xlabel("k (number of guides)")
    axes[0].set_ylabel("Rate (%)")
    axes[0].set_title("Reachability rate vs k\n(dotted = excl. sampling failures from denominator)")
    axes[0].legend(fontsize=8)
    axes[0].set_xscale("log")
    axes[0].xaxis.set_major_formatter(ticker.ScalarFormatter())
    axes[0].set_xticks(k_values)
    axes[0].set_ylim(-5, 105)

    axes[1].set_xlabel("k (number of guides)")
    axes[1].set_ylabel("Rate (%)")
    axes[1].set_title(
        "Failure breakdown vs k\n(solid = unreached; dashed = couldn't sample k guides)"
    )
    axes[1].legend(fontsize=8)
    axes[1].set_xscale("log")
    axes[1].xaxis.set_major_formatter(ticker.ScalarFormatter())
    axes[1].set_xticks(k_values)
    axes[1].set_ylim(-5, 105)

    fig.suptitle(_label, fontsize=14)
    fig.tight_layout()
    plt.show()

## Sampling failure rate vs k

A trial marked `not_enough_samples` means the guide pool for that seed/goal had fewer than `k` entries,
so we could not run the experiment at all. High rates here indicate the strategy's guide pool is too small for large k.

In [ ]:
for _label, df in DATASETS:
    # Sampling failure rate per strategy x k, and per goal x k
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for i, strat in enumerate(strat_names):
        style = strat_style(i)
        strat_df = df.filter(pl.col("strategy") == strat)

        insuf = (
            strat_df.group_by("k")
            .agg(pl.col("not_enough_samples").mean().alias("insuf_rate"))
            .sort("k")
        )
        axes[0].plot(insuf["k"], insuf["insuf_rate"] * 100, label=strat, **style)

    axes[0].set_xlabel("k (number of guides)")
    axes[0].set_ylabel("Insufficient-samples rate (%)")
    axes[0].set_title("Fraction of trials failing due to insufficient samples vs k")
    axes[0].legend(fontsize=8)
    axes[0].set_xscale("log")
    axes[0].xaxis.set_major_formatter(ticker.ScalarFormatter())
    axes[0].set_xticks(k_values)
    axes[0].set_ylim(-5, 105)

    # Per-goal: fraction of k-values where the goal had >= 1 not_enough_samples trial
    goal_insuf = (
        df.group_by("goal", "k")
        .agg(pl.col("not_enough_samples").mean().alias("insuf_rate"))
        .sort("goal", "k")
    )
    goal_insuf_avg = (
        goal_insuf.group_by("goal")
        .agg(pl.col("insuf_rate").mean().alias("avg_insuf"))
        .sort("avg_insuf", descending=True)
    )

    ax = axes[1]
    avg_insuf = goal_insuf_avg["avg_insuf"]
    ax.hist(avg_insuf, bins=20, edgecolor="black", alpha=0.7)
    ax.axvline(
        np.median(avg_insuf), color="red", ls="--", label=f"median={np.median(avg_insuf):.2f}"
    )
    ax.set_xlabel("Per-goal avg insufficient-samples rate (across all k)")
    ax.set_ylabel("Number of goals")
    ax.set_title("Distribution of per-goal sampling failure rate")
    ax.legend()

    fig.suptitle(_label, fontsize=14)
    fig.tight_layout()
    plt.show()

    n_fully_blocked = (avg_insuf == 1.0).sum()
    n_partially_blocked = ((avg_insuf > 0) & (avg_insuf < 1.0)).sum()
    print(f"{_label} — goals with 100% sampling failure across all k: {n_fully_blocked}")
    print(f"{_label} — goals with partial sampling failure: {n_partially_blocked}")

## Box plots: iterations by strategy at each k

In [ ]:
n_k = len(k_values)

for _label, df in DATASETS:
    fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

    for ki, k in enumerate(k_values):
        ax = axes[0][ki]
        box_data = []
        labels = []
        colors = []
        for i, strat in enumerate(strat_names):
            vals = df.filter((pl.col("k") == k) & (pl.col("strategy") == strat))[
                "iters"
            ].drop_nulls()
            if len(vals) > 0:
                box_data.append(vals)
                labels.append(strat)
                colors.append(PALETTE[i % len(PALETTE)])
        if box_data:
            bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
            for patch, color in zip(bp["boxes"], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.6)
        ax.set_title(f"k = {k}")
        ax.set_ylabel("iters" if ki == 0 else "")

    fig.suptitle(f"Iterations by strategy at each k [{_label}]", fontsize=13)
    fig.tight_layout()
    plt.show()

## Box plots: nodes by strategy at each k

Node count includes the guide egraph (i.e. `max(trial_nodes, guide_egraph_nodes)`).

In [ ]:
for _label, df in DATASETS:
    fig, axes = plt.subplots(1, n_k, figsize=(4 * n_k, 5), squeeze=False, sharey=True)

    for ki, k in enumerate(k_values):
        ax = axes[0][ki]
        box_data = []
        labels = []
        colors = []
        for i, strat in enumerate(strat_names):
            vals = df.filter((pl.col("k") == k) & (pl.col("strategy") == strat))[
                "nodes"
            ].drop_nulls()
            if len(vals) > 0:
                box_data.append(vals)
                labels.append(strat)
                colors.append(PALETTE[i % len(PALETTE)])
        if box_data:
            bp = ax.boxplot(box_data, tick_labels=labels, patch_artist=True)
            for patch, color in zip(bp["boxes"], colors):
                patch.set_facecolor(color)
                patch.set_alpha(0.6)
        ax.set_title(f"k = {k}")
        ax.set_ylabel("nodes (incl. guide egraph)" if ki == 0 else "")

    fig.suptitle(f"Nodes by strategy at each k (incl. guide egraph) [{_label}]", fontsize=13)
    fig.tight_layout()
    plt.show()

## Summary table

In [ ]:
from IPython.display import display

for _label, df in DATASETS:
    print(f"\n=== {_label} ===")
    summary = (
        df.group_by("strategy", "k")
        .agg(
            pl.col("iters").mean().round(2).alias("avg_iters"),
            pl.col("nodes").mean().round(0).alias("avg_nodes"),
            pl.col("classes").mean().round(0).alias("avg_classes"),
            pl.col("nodes_per_class").mean().round(2).alias("avg_nodes_per_class"),
            pl.col("total_time").mean().round(4).alias("avg_time_s"),
            pl.col("iters").is_not_null().mean().round(3).alias("reached_rate"),
            pl.col("unreached").mean().round(3).alias("unreached_rate"),
            pl.col("not_enough_samples").mean().round(3).alias("insuf_samples_rate"),
            pl.col("iters").count().alias("n_trials"),
        )
        .sort("k", "strategy")
    )
    display(summary)

## Per-goal analysis

In [ ]:
for _label, df in DATASETS:
    # Per-goal reachability at each k (shared by several plots below)
    # Conditional reach rate excludes sampling-failure trials from the denominator,
    # so it measures "given we could sample k guides, did we reach the goal?".
    goal_reach = (
        df.group_by("goal", "k")
        .agg(
            pl.col("iters").is_not_null().mean().alias("reach_rate"),
            (
                pl.col("iters").is_not_null().sum() / (pl.col("not_enough_samples").not_()).sum()
            ).alias("cond_reach_rate"),
            pl.col("not_enough_samples").mean().alias("insuf_rate"),
        )
        .sort("goal", "k")
    )

    goal_difficulty = (
        goal_reach.group_by("goal")
        .agg(pl.col("cond_reach_rate").mean().alias("avg_reach"))
        .sort("avg_reach")
    )

    print(
        f"{_label} \u2014 {goal_reach.shape[0]} goal\u00d7k combinations, {goal_difficulty.shape[0]} unique goals"
    )

## Goal difficulty distribution

In [ ]:
for _label, df in DATASETS:
    goal_reach = (
        df.group_by("goal", "k")
        .agg(
            pl.col("iters").is_not_null().mean().alias("reach_rate"),
            (
                pl.col("iters").is_not_null().sum() / (pl.col("not_enough_samples").not_()).sum()
            ).alias("cond_reach_rate"),
            pl.col("not_enough_samples").mean().alias("insuf_rate"),
        )
        .sort("goal", "k")
    )
    goal_difficulty = (
        goal_reach.group_by("goal")
        .agg(pl.col("cond_reach_rate").mean().alias("avg_reach"))
        .sort("avg_reach")
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram of per-goal avg reachability
    ax = axes[0]
    avg_reach = goal_difficulty["avg_reach"]
    ax.hist(avg_reach, bins=20, edgecolor="black", alpha=0.7)
    ax.axvline(
        np.median(avg_reach), color="red", ls="--", label=f"median={np.median(avg_reach):.2f}"
    )
    ax.set_xlabel("Average reachability (across all k)")
    ax.set_ylabel("Number of goals")
    ax.set_title("Goal difficulty distribution")
    ax.legend()

    # CDF per k
    ax = axes[1]
    for i, k in enumerate(k_values):
        reach_at_k = goal_reach.filter(pl.col("k") == k).sort("reach_rate")["reach_rate"]
        ax.plot(
            reach_at_k,
            np.linspace(0, 1, len(reach_at_k)),
            label=f"k={k}",
            color=PALETTE[i % len(PALETTE)],
        )
    ax.set_xlabel("Per-goal reachability rate")
    ax.set_ylabel("Fraction of goals (CDF)")
    ax.set_title("CDF of per-goal reachability by k")
    ax.legend()

    fig.suptitle(_label, fontsize=14)
    fig.tight_layout()
    plt.show()

## Goal-level reachability heatmap

In [ ]:
n_strats = len(strat_names)
n_cols_hm = min(3, n_strats)
n_rows_hm = math.ceil(n_strats / n_cols_hm)

for _label, df in DATASETS:
    fig, axes = plt.subplots(
        n_rows_hm,
        n_cols_hm,
        figsize=(6 * n_cols_hm, 10 * n_rows_hm),
        squeeze=False,
    )

    for si, strat in enumerate(strat_names):
        ax = axes[si // n_cols_hm][si % n_cols_hm]

        strat_df = df.filter(pl.col("strategy") == strat)
        goal_reach_s = (
            strat_df.group_by("goal", "k")
            .agg(pl.col("iters").is_not_null().mean().alias("reach_rate"))
            .sort("goal", "k")
        )
        goal_diff_s = (
            goal_reach_s.group_by("goal")
            .agg(pl.col("reach_rate").mean().alias("avg_reach"))
            .sort("avg_reach")
        )
        goal_order_s = goal_diff_s["goal"].to_list()
        goal_idx_s = {g: i for i, g in enumerate(goal_order_s)}

        matrix = np.full((len(goal_order_s), len(k_values)), np.nan)
        for row in goal_reach_s.iter_rows(named=True):
            gi = goal_idx_s[row["goal"]]
            ki = k_values.index(row["k"])
            matrix[gi, ki] = row["reach_rate"]

        im = ax.imshow(matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
        ax.set_xticks(range(len(k_values)))
        ax.set_xticklabels([str(k) for k in k_values])
        ax.set_xlabel("k (number of guides)")
        ax.set_ylabel("Goal (sorted by avg reachability, hardest at top)")
        ax.set_yticks([])
        ax.set_title(f"{strat}")
        fig.colorbar(im, ax=ax, label="Reachability rate", shrink=0.6)

    for si in range(n_strats, n_rows_hm * n_cols_hm):
        axes[si // n_cols_hm][si % n_cols_hm].set_visible(False)

    fig.suptitle(f"Per-goal reachability rate by k [{_label}]", fontsize=14)
    fig.tight_layout()
    plt.show()

## Fraction of goals with at least one successful trial vs k

In [ ]:
for _label, df in DATASETS:
    fig, ax = plt.subplots(figsize=(10, 5))

    for i, strat in enumerate(strat_names):
        style = strat_style(i)
        strat_df = df.filter(pl.col("strategy") == strat)

        any_success_per_goal_k = strat_df.group_by("goal", "k").agg(
            pl.col("iters").is_not_null().any().alias("any_success")
        )
        frac = (
            any_success_per_goal_k.group_by("k")
            .agg(pl.col("any_success").mean().alias("frac_goals"))
            .sort("k")
        )
        ax.plot(frac["k"], frac["frac_goals"] * 100, label=strat, **style)

    ax.set_xlabel("k (number of guides)")
    ax.set_ylabel("Goals with \u2265 1 successful trial (%)")
    ax.set_title(f"Fraction of goals with at least one successful trial vs k [{_label}]")
    ax.legend()
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(k_values)
    ax.set_ylim(-5, 105)
    fig.tight_layout()
    plt.show()

## Diminishing returns: reachability improvement per goal

In [ ]:
for _label, df in DATASETS:
    goal_reach = (
        df.group_by("goal", "k")
        .agg(
            pl.col("iters").is_not_null().mean().alias("reach_rate"),
            (
                pl.col("iters").is_not_null().sum() / (pl.col("not_enough_samples").not_()).sum()
            ).alias("cond_reach_rate"),
            pl.col("not_enough_samples").mean().alias("insuf_rate"),
        )
        .sort("goal", "k")
    )

    # For each goal, plot reachability vs k and color by difficulty cluster
    # Cluster: easy (reach@k=1 > 0.7), hard (reach@k=100 < 0.3), scalable (rest)
    reach_at_k1 = (
        goal_reach.filter(pl.col("k") == k_values[0])
        .select("goal", "reach_rate")
        .rename({"reach_rate": "reach_k1"})
    )
    reach_at_kmax = (
        goal_reach.filter(pl.col("k") == k_values[-1])
        .select("goal", "reach_rate")
        .rename({"reach_rate": "reach_kmax"})
    )
    goal_clusters = reach_at_k1.join(reach_at_kmax, on="goal").with_columns(
        pl.when(pl.col("reach_k1") > 0.7)
        .then(pl.lit("easy"))
        .when(pl.col("reach_kmax") < 0.3)
        .then(pl.lit("hard"))
        .otherwise(pl.lit("scalable"))
        .alias("cluster")
    )

    cluster_colors = {"easy": "green", "scalable": "steelblue", "hard": "red"}
    cluster_counts = goal_clusters.group_by("cluster").len().sort("cluster")
    print(f"{_label} \u2014 goal clusters:")
    print(cluster_counts)

    fig, ax = plt.subplots(figsize=(10, 6))
    for cluster, color in cluster_colors.items():
        goals_in = goal_clusters.filter(pl.col("cluster") == cluster)["goal"].to_list()
        for gi, g in enumerate(goals_in):
            row = goal_reach.filter(pl.col("goal") == g).sort("k")
            ax.plot(
                row["k"],
                row["reach_rate"] * 100,
                color=color,
                alpha=0.4,
                linewidth=1,
                label=cluster if gi == 0 else None,
            )

    ax.set_xlabel("k (number of guides)")
    ax.set_ylabel("Reachability (%)")
    ax.set_title(
        f"Per-goal reachability vs k (easy: reach@k={k_values[0]}>70%, hard: reach@k={k_values[-1]}<30%) [{_label}]"
    )
    ax.set_xscale("log")
    ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
    ax.set_xticks(k_values)
    ax.set_ylim(-5, 105)
    ax.legend()
    fig.tight_layout()
    plt.show()

## Reachability tradeoff

Node count is `max(trial_nodes, guide_egraph_nodes)` and time includes `guide_eqsat_time`, so both reflect the true cost of the guided run.

In [ ]:
for _label, df in DATASETS:
    # Per strategy×k: avg nodes (reached only) vs reachability
    tradeoff = (
        df.group_by("strategy", "k")
        .agg(
            pl.col("iters").is_not_null().mean().alias("reachability"),
            pl.col("nodes").mean().alias("avg_nodes"),
            pl.col("total_time").mean().alias("avg_time"),
        )
        .sort("strategy", "k")
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, strat in enumerate(strat_names):
        style = strat_style(i)
        sub = tradeoff.filter(pl.col("strategy") == strat)
        ks = sub["k"].to_list()

        # Nodes vs reachability
        ax = axes[0]
        ax.plot(sub["avg_nodes"], sub["reachability"] * 100, **style, label=strat)
        for k_val, x, y in zip(ks, sub["avg_nodes"].to_list(), sub["reachability"].to_list()):
            ax.annotate(
                f"k={k_val}", (x, y * 100), textcoords="offset points", xytext=(5, 5), fontsize=7
            )

        # Time vs reachability
        ax = axes[1]
        ax.plot(sub["avg_time"] * 1000, sub["reachability"] * 100, **style, label=strat)
        for k_val, x, y in zip(ks, sub["avg_time"].to_list(), sub["reachability"].to_list()):
            ax.annotate(
                f"k={k_val}",
                (x * 1000, y * 100),
                textcoords="offset points",
                xytext=(5, 5),
                fontsize=7,
            )

    axes[0].set_xlabel("Avg nodes, incl. guide egraph (all trials)")
    axes[0].set_ylabel("Reachability (%)")
    axes[0].set_title("Reachability vs node cost (incl. guide egraph)")
    axes[0].legend()

    axes[1].set_xlabel("Avg time, incl. guide eqsat (ms)")
    axes[1].set_ylabel("Reachability (%)")
    axes[1].set_title("Reachability vs time cost (incl. guide eqsat)")
    axes[1].legend()

    fig.suptitle(_label, fontsize=14)
    fig.tight_layout()
    plt.show()

## Wins Above Replacement (WAR)

WAR measures how much better each strategy is compared to the full eqsat baseline (no guides).

- **nodes WAR**: baseline nodes − mean guided nodes (positive = guided uses fewer nodes; guided nodes = `max(trial_nodes, guide_egraph_nodes)`)
- **time WAR**: baseline time − mean guided time (positive = guided is faster; guided time includes `guide_eqsat_time`)

Both are computed per strategy × k.

In [ ]:
for _label, df in DATASETS:
    # ── Wins Above Replacement ────────────────────────────────────────────
    # WAR = baseline_value - mean_guided_value  (positive = guided is better)
    war_rows = []
    for strat in strat_names:
        agg = (
            df.filter(pl.col("strategy") == strat)
            .group_by("k")
            .agg(
                pl.col("nodes").mean().alias("mean_nodes"),
                pl.col("total_time").mean().alias("mean_time"),
            )
            .sort("k")
        )
        # Best: per seed×k take min over trials, then average across seeds
        best_agg = (
            df.filter(pl.col("strategy") == strat)
            .group_by("k", "seed")
            .agg(
                pl.col("nodes").min().alias("seed_best_nodes"),
                pl.col("total_time").min().alias("seed_best_time"),
            )
            .group_by("k")
            .agg(
                pl.col("seed_best_nodes").mean().alias("best_nodes"),
                pl.col("seed_best_time").mean().alias("best_time"),
            )
            .sort("k")
        )
        for row, best_row in zip(agg.iter_rows(named=True), best_agg.iter_rows(named=True)):
            war_rows.append(
                {
                    "strategy": strat,
                    "k": row["k"],
                    "nodes_war": baselines[strat]["nodes"] - row["mean_nodes"],
                    "time_war": baselines[strat]["total_time"] - row["mean_time"],
                    "best_nodes_war": baselines[strat]["nodes"] - best_row["best_nodes"],
                    "best_time_war": baselines[strat]["total_time"] - best_row["best_time"],
                }
            )

    war_df = pl.DataFrame(war_rows).sort("strategy", "k")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, strat in enumerate(strat_names):
        style = strat_style(i)
        sub = war_df.filter(pl.col("strategy") == strat).sort("k")
        ks = sub["k"]

        axes[0].plot(ks, sub["nodes_war"], label=strat, **style)
        axes[0].plot(
            ks,
            sub["best_nodes_war"],
            color=style["color"],
            linestyle="dotted",
            linewidth=1.5,
            label=f"{strat} best (min)",
        )

        axes[1].plot(ks, sub["time_war"] * 1000, label=strat, **style)
        axes[1].plot(
            ks,
            sub["best_time_war"] * 1000,
            color=style["color"],
            linestyle="dotted",
            linewidth=1.5,
            label=f"{strat} best (min)",
        )

    for ax in axes:
        ax.axhline(0, color="black", linewidth=0.8, linestyle="-", alpha=0.5)
        ax.set_xscale("log")
        ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
        ax.set_xticks(k_values)
        ax.set_xlabel("k (number of guides)")
        ax.legend(fontsize=7)

    axes[0].set_ylabel("nodes WAR  (baseline - guided, higher is better)")
    axes[0].set_title("Nodes WAR vs k")
    axes[1].set_ylabel("time WAR  (ms, baseline - guided, higher is better)")
    axes[1].set_title("Time WAR vs k")

    fig.suptitle(_label, fontsize=14)
    fig.tight_layout()
    plt.show()

    print(war_df)